# Sentiment Agent Demo

The Sentiment Agent is the second specialist, now registered alongside Finance in the boss
agent's roster (`backend/agents/boss/registry.py`).

## Its 7 tools

| Tool | Computes | Data reality |
|---|---|---|
| `search_reviews` | Keyword match over review text | Proxy for real semantic search — Vector Search (RAG layer, build order step 6) isn't built yet |
| `sentiment_score_by_product` | Avg `review_score` (1-5) per product, top/bottom N | `review_score` is already a direct sentiment signal in the data — no NLP needed here |
| `flag_negative_trend` | Monthly share of score <=2 reviews, trend direction | Computed directly |
| `extract_common_complaints` | Word-frequency on negative review text | Raw Portuguese, basic stopword filtering only — real NLP topic extraction is the NLP/DL layer (step 2, not built) |
| `review_response_time_correlation` | Correlation between survey-answer latency and score | **Not seller response time** — Olist's `review_answer_timestamp` is when the *customer* answered Olist's satisfaction survey. No seller-response field exists in this dataset. |
| `sentiment_by_region` | Avg score by customer state | Computed directly |
| `photo_review_analysis` | Correlation between product *listing* photo count and score | Olist reviews have no photo attachments — uses `products.product_photos_qty` as the closest proxy |

## The Portuguese-text problem (flagged in CLAUDE.md section 2)

Review text in this dataset is almost entirely Portuguese. Two tools above touch raw review
text (`search_reviews`, `extract_common_complaints`) and both are explicitly built as
proxies right now, not real NLP/semantic capabilities — matching the project's data-honesty
pattern already established in the Finance Agent (flag limitations, don't fabricate). Proper
translation/embedding is scoped to the RAG layer, later in the build order.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from backend.agents.sentiment import SentimentAgent
from backend.db import DatabricksClient

db = DatabricksClient()
agent = SentimentAgent(db)

## Run the agent against live Delta tables

In [2]:
briefing = agent.run("How are customers feeling about our products lately?")

print(f"Agent: {briefing.agent.value}")
print(f"Execution time: {briefing.execution_time_ms:.0f}ms")
print(f"Tool calls: {len(briefing.tool_calls)} ({sum(1 for t in briefing.tool_calls if t.success)} succeeded)")

Agent: Sentiment
Execution time: 16716ms
Tool calls: 7 (7 succeeded)


## Inspect the findings

In [3]:
for f in briefing.findings:
    print(f"[{f.severity.upper()}] (confidence {f.confidence:.2f}) {f.claim}")
    print(f"  source: {f.source}\n")

[INFO] (confidence 0.40) Keyword search for 'How are customers feeling about our products lately?' matched 0 reviews
  source: search_reviews

[WARNING] (confidence 0.85) Lowest-rated product (min 5 reviews) is 0c40401eba358c9ef2ff2df80b6eab52 at 1.0/5 (5 reviews)
  source: sentiment_score_by_product

[INFO] (confidence 0.85) Negative review share (score <=2) is flat across the last 12 months
  source: flag_negative_trend

[INFO] (confidence 0.50) Top recurring terms in negative reviews (Portuguese, 10890 reviews): produto, recebi, comprei, veio, entrega
  source: extract_common_complaints

[INFO] (confidence 0.60) Survey-answer latency vs. review score correlation is 0.007 across 99132 reviews
  source: review_response_time_correlation

[INFO] (confidence 0.80) Sentiment ranges from 4.19/5 in AP to 3.61/5 in RR
  source: sentiment_by_region

[INFO] (confidence 0.55) Product listing photo count vs. review score correlation is 0.023 across 110773 reviews
  source: photo_review_analysis


## Through the boss agent

Now with two specialists registered, the boss agent's `select_specialists` step has a real
choice to make — not just "Finance or nothing" but "Finance, Sentiment, both, or neither".

In [4]:
from backend.agents.boss import BossAgent

boss = BossAgent(db)
rec = boss.run("Are customers happy, and is that reflected in the numbers?")

print(f"Agents invoked: {[a.value for a in rec.agents_invoked]}")
print(f"Overall confidence: {rec.confidence_overall}")
print(f"\nSynthesis:\n{rec.synthesis}")

if rec.dissents:
    print("\nDissents:")
    for d in rec.dissents:
        print(f"  - {[a.value for a in d.agents_involved]}: {d.summary}")

Agents invoked: ['Sentiment', 'Finance']
Overall confidence: 0.7

Synthesis:
Board Memo: Customer Happiness and Financial Indicators

**Summary**
Finance’s analysis (calculate_margin_trend, revenue_concentration) shows an improving contribution margin and diversified revenue streams, suggesting healthy profitability. However, Sentiment’s review analytics (sentiment_score_by_product, flag_negative_trend) reveal that the lowest-rated product has a 1.0/5 score and that negative review share has remained flat over the past year, indicating persistent customer dissatisfaction in specific areas.

**Key Findings**
1. **Profitability** – Finance’s calculate_margin_trend tool reports an upward trend in contribution margin over the last 12 months (confidence 0.75). Cash flow forecasts (cash_flow_forecast) project next month’s cash‑in at $1,177,859.93 (confidence 0.55). Revenue concentration is low (revenue_concentration) with the top 10 sellers accounting for 13.2% of revenue (confidence 0.9). 


## What's pending

- Real semantic search for `search_reviews` once Vector Search is built (step 6)
- Real complaint-theme extraction (translation + NLP) once the NLP/DL layer exists (step 2)
- 5 more specialists before the roster is complete